# Existing Charging Stations — Cleaning + EDA Pipeline

Goal: build a reliable baseline of **existing interurban charging stations** for the datathon optimization pipeline.

This notebook is organized in this order:
1. Ingest + parse XML
2. Normalize into 3 tables (`sites`, `refill_points`, `connectors`)
3. Preprocess (missing values, text normalization, stable keys, power standardization)
4. Clean (coordinate validation, deduplication, interurban filtering)
5. EDA (coverage, operators, power, station types, missingness, duplicates)


## Data dictionary (target schema)

### Table 1 — `sites`
- `site_key`
- `site_name`
- `operator_name`
- `lat`, `lon`
- `postcode`, `address`
- `type_of_site`, `service_facility_type`
- `last_updated`
- `interurban_flag`
- `coord_valid`

### Table 2 — `refill_points`
- `site_key`
- `station_index`
- `refill_point_index`
- `charging_mode`
- `connector_count`

### Table 3 — `connectors`
- `site_key`
- `connector_index`
- `connector_type`
- `connector_format`
- `max_power_kw`
- `voltage`
- `max_current`


In [ ]:
# Standard-library only version (portable in restricted environments)
import re
import math
import json
import hashlib
from datetime import datetime, timezone
from collections import Counter
import xml.etree.ElementTree as ET

try:
    import requests
except Exception as exc:
    raise ImportError('requests is required for remote XML download.') from exc


In [ ]:
# -----------------------------
# 1) Load XML (remote or local)
# -----------------------------
XML_URL = 'https://infocar.dgt.es/datex2/v3/miterd/EnergyInfrastructureTablePublication/electrolineras.xml'
LOCAL_XML_FALLBACK = None  # Example: '/absolute/path/electrolineras.xml'


def load_xml_root(xml_url=XML_URL, local_fallback=LOCAL_XML_FALLBACK, timeout=60):
    if local_fallback:
        return ET.parse(local_fallback).getroot(), {'source': 'local_file', 'path': local_fallback}

    response = requests.get(xml_url, timeout=timeout)
    response.raise_for_status()
    return ET.fromstring(response.content), {'source': 'remote_url', 'url': xml_url}


root, source_info = load_xml_root()
print('Loaded XML from:', source_info)


In [ ]:
# --------------------------------------------
# 2) Parse XML and flatten into normalized rows
# --------------------------------------------

def localname(tag):
    return tag.split('}', 1)[-1] if '}' in tag else tag


def norm_text(x):
    if x is None:
        return None
    s = str(x).strip()
    if not s:
        return None
    s = re.sub(r'\s+', ' ', s)
    return s


def first_text_by_tag(elem, candidate_tags):
    for child in elem.iter():
        if localname(child.tag) in candidate_tags:
            v = norm_text(child.text)
            if v is not None:
                return v
    return None


def parse_float(value):
    if value is None:
        return None
    s = str(value).replace(',', '.').strip()
    s = re.sub(r'[^0-9+\-\.eE]', '', s)
    if s in {'', '+', '-', '.', '+.', '-.'}:
        return None
    try:
        return float(s)
    except ValueError:
        return None


def power_to_kw(raw_power):
    # Handles values like "150", "150 kW", "50000 W", "50kW"
    if raw_power is None:
        return None
    s = str(raw_power).strip().lower().replace(',', '.')
    num = parse_float(s)
    if num is None:
        return None

    if 'mw' in s:
        return num * 1000.0
    if 'w' in s and 'kw' not in s:
        return num / 1000.0
    return num


def is_valid_coord(lat, lon):
    if lat is None or lon is None:
        return False
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)


def infer_interurban(type_of_site, service_facility_type, address):
    # Conservative heuristic; adjust based on domain review.
    txt = ' | '.join([x for x in [type_of_site, service_facility_type, address] if x]).lower()

    interurban_markers = [
        'motorway', 'highway', 'autovia', 'autopista', 'carretera',
        'service area', 'area de servicio', 'interurban', 'n-', 'a-', 'ap-'
    ]
    urban_markers = [
        'urban', 'city center', 'downtown', 'municipal',
        'parking urbano', 'centro comercial', 'residential'
    ]

    if any(m in txt for m in urban_markers):
        return False
    if any(m in txt for m in interurban_markers):
        return True
    return None




def parse_iso8601_utc(raw_ts):
    if raw_ts is None:
        return None
    s = norm_text(raw_ts)
    if s is None:
        return None
    try:
        dt = datetime.fromisoformat(s.replace('Z', '+00:00'))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc).isoformat()
    except Exception:
        return s


def stable_site_key(national_identifier, site_name, lat, lon, address):
    if national_identifier:
        return f'ni::{national_identifier.strip().lower()}'

    parts = [
        norm_text(site_name) or '',
        '' if lat is None else f'{lat:.6f}',
        '' if lon is None else f'{lon:.6f}',
        norm_text(address) or ''
    ]
    raw = '||'.join(parts).lower()
    digest = hashlib.sha1(raw.encode('utf-8')).hexdigest()[:16]
    return f'hash::{digest}'


site_nodes = [x for x in root.iter() if localname(x.tag) == 'energyInfrastructureSite']
print('Raw site nodes:', len(site_nodes))

sites = []
refill_points = []
connectors = []

for site_idx, site in enumerate(site_nodes):
    national_identifier = first_text_by_tag(site, ['nationalIdentifier'])
    site_name = first_text_by_tag(site, ['name'])
    operator_name = first_text_by_tag(site, ['operator'])
    lat = parse_float(first_text_by_tag(site, ['latitude']))
    lon = parse_float(first_text_by_tag(site, ['longitude']))
    postcode = first_text_by_tag(site, ['postcode'])
    address = first_text_by_tag(site, ['addressLine', 'address'])
    type_of_site = first_text_by_tag(site, ['typeOfSite'])
    service_facility_type = first_text_by_tag(site, ['serviceFacilityType'])
    last_updated = parse_iso8601_utc(first_text_by_tag(site, ['versionTime', 'publicationTime', 'lastUpdated']))

    site_key = stable_site_key(national_identifier, site_name, lat, lon, address)

    site_row = {
        'site_key': site_key,
        'national_identifier': national_identifier,
        'site_name': site_name,
        'operator_name': operator_name,
        'lat': lat,
        'lon': lon,
        'postcode': postcode,
        'address': address,
        'type_of_site': type_of_site,
        'service_facility_type': service_facility_type,
        'last_updated': last_updated,
    }
    sites.append(site_row)

    station_nodes = [x for x in site.iter() if localname(x.tag) == 'energyInfrastructureStation']
    refill_nodes = [x for x in site.iter() if localname(x.tag) == 'refillPoint']
    connector_nodes = [x for x in site.iter() if localname(x.tag) == 'connector']

    for refill_idx, rp in enumerate(refill_nodes):
        charging_mode = first_text_by_tag(rp, ['chargingMode'])
        connector_count = len([x for x in rp.iter() if localname(x.tag) == 'connector'])
        refill_points.append({
            'site_key': site_key,
            'station_index': 0 if station_nodes else None,
            'refill_point_index': refill_idx,
            'charging_mode': charging_mode,
            'connector_count': connector_count,
        })

    for conn_idx, conn in enumerate(connector_nodes):
        connector_type = first_text_by_tag(conn, ['connectorType'])
        connector_format = first_text_by_tag(conn, ['connectorFormat'])
        max_power_raw = first_text_by_tag(conn, ['maxPowerAtSocket'])
        voltage = parse_float(first_text_by_tag(conn, ['voltage']))
        max_current = parse_float(first_text_by_tag(conn, ['maximumCurrent']))

        connectors.append({
            'site_key': site_key,
            'connector_index': conn_idx,
            'connector_type': connector_type,
            'connector_format': connector_format,
            'max_power_kw': power_to_kw(max_power_raw),
            'voltage': voltage,
            'max_current': max_current,
        })

print('Parsed rows -> sites:', len(sites), '| refill_points:', len(refill_points), '| connectors:', len(connectors))


In [ ]:
# ---------------------------------
# 3) Cleaning and quality operations
# ---------------------------------

def round_coord(lat, lon, decimals=6):
    if lat is None or lon is None:
        return None, None
    return round(lat, decimals), round(lon, decimals)


def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1 = math.radians(lat1)
    p2 = math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


for s in sites:
    s['coord_valid'] = is_valid_coord(s['lat'], s['lon'])
    s['interurban_flag'] = infer_interurban(s['type_of_site'], s['service_facility_type'], s['address'])
    s['site_name_norm'] = (norm_text(s['site_name']) or '').lower()
    s['operator_name_norm'] = (norm_text(s['operator_name']) or '').lower()


exact_key_to_best = {}
for s in sites:
    lat_r, lon_r = round_coord(s['lat'], s['lon'], decimals=6)
    dedup_key = (
        s['national_identifier'] or '',
        s['site_name_norm'],
        s['operator_name_norm'],
        lat_r,
        lon_r,
        (norm_text(s['address']) or '').lower(),
    )
    existing = exact_key_to_best.get(dedup_key)
    if existing is None:
        exact_key_to_best[dedup_key] = s
        continue

    cur_score = (int(s['coord_valid']), len(s.get('address') or ''))
    old_score = (int(existing['coord_valid']), len(existing.get('address') or ''))
    if cur_score > old_score:
        exact_key_to_best[dedup_key] = s

sites_dedup_exact = list(exact_key_to_best.values())


NEAR_DUPLICATE_THRESHOLD_KM = 0.1  # 100m: flags likely duplicated station points for manual review

near_duplicate_pairs = []
valid_idx = [i for i, s in enumerate(sites_dedup_exact) if s['coord_valid']]
for ii in range(len(valid_idx)):
    i = valid_idx[ii]
    a = sites_dedup_exact[i]
    for jj in range(ii + 1, len(valid_idx)):
        j = valid_idx[jj]
        b = sites_dedup_exact[j]
        if a['operator_name_norm'] != b['operator_name_norm']:
            continue
        d_km = haversine_km(a['lat'], a['lon'], b['lat'], b['lon'])
        if d_km <= NEAR_DUPLICATE_THRESHOLD_KM:
            near_duplicate_pairs.append((a['site_key'], b['site_key'], round(d_km, 4)))


sites_interurban = [s for s in sites_dedup_exact if s['coord_valid'] and s['interurban_flag'] is True]
site_keys_interurban = {s['site_key'] for s in sites_interurban}
refill_points_interurban = [r for r in refill_points if r['site_key'] in site_keys_interurban]
connectors_interurban = [c for c in connectors if c['site_key'] in site_keys_interurban]

print('Sites raw:', len(sites))
print('Sites after exact dedup:', len(sites_dedup_exact))
print('Sites interurban + valid coords:', len(sites_interurban))
print('Potential near-duplicate pairs (review manually):', len(near_duplicate_pairs))


In [ ]:
# -------------------------
# 4) Focused EDA (concise)
# -------------------------

def top_counts(values, n=10):
    c = Counter([v for v in values if v])
    return c.most_common(n)


def pct(part, whole):
    return 0.0 if whole == 0 else 100.0 * part / whole


n_sites_raw = len(sites)
n_sites_clean = len(sites_interurban)
n_refill_clean = len(refill_points_interurban)
n_connectors_clean = len(connectors_interurban)

print('--- Core counts ---')
print(f'Sites (raw): {n_sites_raw}')
print(f'Sites (clean interurban): {n_sites_clean} ({pct(n_sites_clean, n_sites_raw):.1f}% of raw)')
print(f'Refill points (clean): {n_refill_clean}')
print(f'Connectors (clean): {n_connectors_clean}')

fields = ['site_name', 'operator_name', 'lat', 'lon', 'postcode', 'address', 'type_of_site', 'service_facility_type']
missing = {}
for f in fields:
    missing[f] = sum(1 for s in sites_interurban if s.get(f) in [None, ''])

print()
print('--- Missingness (clean sites) ---')
for f in fields:
    print(f'{f}: {missing[f]} missing ({pct(missing[f], n_sites_clean):.1f}%)')

print()
print('--- Top operators ---')
for op, cnt in top_counts([s.get('operator_name_norm') for s in sites_interurban], n=15):
    print(f'{op}: {cnt}')

print()
print('--- Top site/service types ---')
type_pairs = [f"{(s.get('type_of_site') or 'NA')} | {(s.get('service_facility_type') or 'NA')}" for s in sites_interurban]
for t, cnt in top_counts(type_pairs, n=15):
    print(f'{t}: {cnt}')

powers = [c['max_power_kw'] for c in connectors_interurban if c.get('max_power_kw') is not None]
powers_sorted = sorted(powers)

print()
print('--- Power distribution (kW) ---')
if powers_sorted:
    def quantile(vs, p):
        idx = int((len(vs)-1) * p)
        return vs[idx]

    print('count:', len(powers_sorted))
    print('min:', round(powers_sorted[0], 2))
    print('p25:', round(quantile(powers_sorted, 0.25), 2))
    print('median:', round(quantile(powers_sorted, 0.50), 2))
    print('p75:', round(quantile(powers_sorted, 0.75), 2))
    print('max:', round(powers_sorted[-1], 2))
else:
    print('No valid power values found.')

coords = [(s['lat'], s['lon']) for s in sites_interurban]
print()
print('--- Geographic spread ---')
if coords:
    lats = [x[0] for x in coords]
    lons = [x[1] for x in coords]
    print('lat range:', (min(lats), max(lats)))
    print('lon range:', (min(lons), max(lons)))
else:
    print('No valid interurban coordinates.')

print()
print('--- Duplicate analysis ---')
print('Exact duplicates removed:', n_sites_raw - len(sites_dedup_exact))
print('Near-duplicate candidate pairs:', len(near_duplicate_pairs))


## Optional next step: snap stations to nearest road segment

Recommended when your road network graph is ready:
- Input: clean `sites_interurban` + road nodes/edges
- For each site, compute nearest road point and distance
- Keep both original and snapped coordinates
- Add quality guardrail (e.g., reject snap if distance > 2 km)

This step improves alignment with your **Network Layer** and avoids off-road station artifacts.


In [ ]:
# --------------------------------------------------
# 5) Export cleaned outputs for downstream pipeline
# --------------------------------------------------
EXPORT_PREFIX = 'charging_clean'


def export_jsonl(path, rows):
    with open(path, 'w', encoding='utf-8') as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')


export_jsonl(f'{EXPORT_PREFIX}_sites.jsonl', sites_interurban)
export_jsonl(f'{EXPORT_PREFIX}_refill_points.jsonl', refill_points_interurban)
export_jsonl(f'{EXPORT_PREFIX}_connectors.jsonl', connectors_interurban)

qa_summary = {
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'source': source_info,
    'counts': {
        'sites_raw': len(sites),
        'sites_after_exact_dedup': len(sites_dedup_exact),
        'sites_interurban_clean': len(sites_interurban),
        'refill_points_interurban_clean': len(refill_points_interurban),
        'connectors_interurban_clean': len(connectors_interurban),
        'near_duplicate_pairs': len(near_duplicate_pairs),
    }
}

with open(f'{EXPORT_PREFIX}_qa_summary.json', 'w', encoding='utf-8') as f:
    json.dump(qa_summary, f, indent=2, ensure_ascii=False)

print('Exported cleaned tables and QA summary.')


## Notes for report justification (brief)

- **Interurban filtering** is heuristic-first, then should be calibrated with road-context joins.
- **Coordinate validation** removes records that cannot be mapped reliably.
- **Deduplication** uses exact normalized identity + near-duplicate review candidates.
- **Power standardization** converts mixed textual formats into `kW` for consistent analytics.

This cleaned baseline is ready to plug into:
- gap analysis vs existing stations,
- candidate generation constraints,
- and grid/friction downstream layers.
